# Lesson 04 - ကိရိယာအသုံးပြုမှု ဒီဇိုင်းပုံစံ

ဤသင်ခန်းစာတွင် Microsoft Agent Framework (Python) အသုံးပြု၍ AI အေးဂျင့်များအတွက် **ကိရိယာအသုံးပြုမှု** ဒီဇိုင်းပုံစံကို သင်ယူမည် ဖြစ်သည်။ အောက်ပါအမျိုးအစားများကို ဖော်ပြပါသည်-

- `@tool` ဒီကိုရေးတာနှင့် အမျိုးအစားသတ်မှတ်ထားသော ပါရာမီတာများဖြင့် ကိရိယာများကို သတ်မှတ်ခြင်း
- မော်ဒယ်သည် ကိရိယာတစ်ခုစီ ဘာလုပ်ဆောင်သည်ကို သိရှိစေရန် ကိရိယာစကီးမာများ ပံ့ပိုးပေးခြင်း
- `approval_mode` ဖြင့် ကိရိယာ အကောင်အထည်ဖော်ခြင်းကို ထိန်းချုပ်ခြင်း
- Pydantic မော်ဒယ်များနှင့် `response_format` ဖြင့် **ဖွဲ့စည်းထားသော output** ပြန်လည်ထုတ်ပေးခြင်း

ဖြစ်ရပ်အကြောင်းအရာမှာ **ခရီးသွား စီစဉ်မှုပေးသော အေးဂျင့်** တစ်ယောက်ဖြစ်ပြီး မည်သည့်နေရာများကို လှမ်းရှာနိုင်ခြင်း၊ ရရှိနိုင်မှုကိုစစ်ဆေးခြင်းနှင့် ဧည့်သည်လေယာဉ်အချက်အလက်များကို ရယူနိုင်သည်။


## စနစ်တက်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool အကြောင်းပြောကြားခြင်း

`@tool` decorator သည် ရိုးရှင်းသော Python function ကို စက်တစ်လုံးက ဖုန်းခေါ်နိုင်သော tool တစ်ခုအဖြစ် ပြောင်းလဲပေးသည်။
အဓိကအချက်များ -

- **docstring** သည် မော်ဒယ်တွေ မြင်ရသော tool ဖော်ပြချက် ဖြစ်သည်။
- **Type annotations** (ဖော်ပြချက်များပါရှိသည့် `Annotated` အပါအဝင်) သည် tool schema ကို သတ်မှတ်ပေးသည်။
- `approval_mode` သည် ဖုန်းခေါ်ခြင်းတိုင်းကို အသုံးပြုသူ သဘောတူရန် လိုအပ်မည်ကို ထိန်းချုပ်ပေးသည်။


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Tools များစွာပါသော Agent တစ်ခု ဖန်တီးခြင်း

user ရဲ့ မေးခွန်းကို ဖြေဆိုရန် မော်ဒယ်က လိုအပ်သည့် tools များကို မည်သည့် tools မှာမဆို ဖိတ်ခေါ်နိုင်ရန် client သို့ tool သုံးခုစလုံးကို ပေးပို့ပါ။


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ကိရိယာများနှင့် အစီအစဉ်ဖွဲ့ ထုတ်လွှင့်ချက်

`response_format` ကို Pydantic မော်ဒယ်တစ်ခုအဖြစ် သတ်မှတ်ခြင်းအားဖြင့်၊ agent သည် အခမဲ့ပုံစံစာသားမဟုတ်ဘဲ ကောင်းမွန်စွာ စနစ်တကျ JSON အရာဝတ္တုတစ်ခုကို ပြန်ပေးရန် မရှိမဖြစ်လိုအပ်ပါသည်။ ၎င်းသည် အောက်ဆက်မည့် ကုဒ်များက ရလဒ်ကို အလိုအလျောက် စားသုံးရန် လိုအပ်သောအခါ အသုံးဝင်သည်။


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Tool Approval Patterns

`@tool` တွင်ရှိသော `approval_mode` ပါရာမီတာသည် ကိရိယာအား အလုပ်လုပ်ရန် မတိုင်မီ လူအတည်ပြုချက် လိုအပ်မလား ဆိုတာကို ထိန်းချုပ်သည်။

| Mode | အသွင်အပြင် |
|---|---|
| `"never_require"` | ကိရိယာသည် တစ်ချက်ပြိုင်သည့် မတိုင်မီ လူအသုံးပြုသူ၏ အတည်ပြုချက်မလိုအပ်ဘဲ အလိုအလျောက် အလုပ်လုပ်သည်။ |
| `"always_require"` | မည်သည့်ခေါ်ယူမှုမျှမဆို အလုပ်လုပ်မည်မတိုင်မီ အသုံးပြုသူ၏ အတည်ပြုချက် လိုအပ်သည်။ |

သက်ရောက်မှုရှိသော (ဥပမာ- လေကြောင်းလက်မှတ်မှာ၊ ခရက်ဒစ်ကတ် ကွက်ခံခြင်း) ကိရိယာများအတွက် `"always_require"` ကို သုံးပါ၊ ဒါမှ လူအတည်ပြုချက် နေထိုင်နိုင်သည်။


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## အနှစ်ချုပ်

ဒီသင်ခန်းစာမှာ သင်ယူခဲ့တာတွေကတော့ -

1. **@tool** decorator ကို အသုံးပြုပြီး typed parameters နဲ့ docstrings တွေကို tool schema အနေနဲ့ သတ်မှတ်ခြင်း။
2. **အတန်းလိုက်ခေါ်နိုင်ဖို့** tools များကို ပေါင်းစပ်ဖန်တီးခြင်း။
3. Pydantic model ကို `response_format` အဖြစ်ပေးပြီး **ဖွဲ့စည်းထားသော output ကို ပြန်ပေးပို့ခြင်း**။
4. အရေးကြီး လုပ်ငန်းစဉ်များအတွက် လူတစ်ဦး ပါဝင်စောင့်ကြည့်ရုံအနေဖြင့် `approval_mode` ဖြင့် **tool ကို အတည်ပြုခြင်းကို ထိန်းချုပ်ခြင်း**။

ဒီနည်းလမ်းတွေက တိကျနှင့် ထိရောက်စွာ စနစ်ပြုပြင်ထားတဲ့ agent များကို ထူထောင်ရန် အခြေခံ ချဉ်းကပ်မှုများဖြစ်စေပါတယ်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
